## Import

In [6]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [2]:
from datasets.Student_t import MultivariateStudentT

d = 32
dim_x = dim_y = d//2
rho = 0.8
df = 1
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  9.52
X torch.Size([10000, 16]) Y torch.Size([10000, 16])


## Hyperaparams

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [5]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.9997274875640869 loss val= 1.0054903030395508 best val loss= 1.0054903030395508 best t= 0
finished: t= 76 loss= 0.10736124962568283 loss val= 0.1557173728942871 best val loss= 0.10662606358528137 best t= 75
finished: t= 152 loss= 0.08184237778186798 loss val= 0.10351350903511047 best val loss= 0.0941542312502861 best t= 137
finished: t= 228 loss= 0.09036116302013397 loss val= 0.11268678307533264 best val loss= 0.07670362293720245 best t= 227
finished: t= 304 loss= 0.10653901845216751 loss val= 0.10716310143470764 best val loss= 0.07365186512470245 best t= 270
finished: t= 380 loss= 0.08798778057098389 loss val= 0.10270106792449951 best val loss= 0.07039584219455719 best t= 354
finished: t= 456 loss= 0.058862391859292984 loss val= 0.07202732563018799 best val loss= 0.06508954614400864 best t= 454
finished: t= 532 loss= 0.07914622128009796 loss val= 0.07932211458683014 best val loss= 0.0632237121462822 best t= 463
finished: t= 608 loss= 0.09530013054609299 loss val

In [11]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 73.04354858398438 loss val= 237.21078491210938 best val loss= 237.21078491210938 best t= 0
finished: t= 76 loss= 2.4219236373901367 loss val= 3.350947141647339 best val loss= 1.2304739952087402 best t= 71
finished: t= 152 loss= 7.492395877838135 loss val= 7.131320953369141 best val loss= 0.45309680700302124 best t= 135
finished: t= 228 loss= 5.330896377563477 loss val= 5.758667945861816 best val loss= 0.45309680700302124 best t= 135


true MI: 9.517696203706947
est MI: 1.0890768766403198


In [7]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 1.0133622884750366 loss val= 5.556449890136719 best val loss= 5.556449890136719 best t= 0
finished: t= 126 loss= 0.06633377075195312 loss val= 5.71952486038208 best val loss= 4.558210849761963 best t= 123
finished: t= 252 loss= 0.09852209687232971 loss val= 7.489372730255127 best val loss= 4.558210849761963 best t= 123


finished: t= 0 loss= 1.0340769290924072 loss val= 5.411300182342529 best val loss= 5.411300182342529 best t= 0
finished: t= 126 loss= 0.4166463315486908 loss val= 4.618242263793945 best val loss= 4.346778392791748 best t= 103
finished: t= 252 loss= 0.10722757130861282 loss val= 4.186248779296875 best val loss= 3.7264862060546875 best t= 228


finished: t= 0 loss= 131.53204345703125 loss val= 132.1867218017578 best val loss= 132.1867218017578 best t= 0
finished: t= 101 loss= 33.826786041259766 loss val= 34.37605285644531 best val loss= 34.37439727783203 best t= 100
finished: t= 202 loss= 33.6404380